# Serial 3D reconstruction
Prepare `series.json`: an ordered list of pair-manifest paths, one for each consecutive section pair. The first section defines the reference frame. Every shared section must have identical IDs and coordinates across adjacent manifests.
For Deep-STARmap use the MERFISH selector. Its historical calibration of 100 units/100 µm is a numerical convention, not independently verified physical calibration. The z spacing is specified separately.


## Data and local paths
Data are not uploaded to GitHub. Download sources and preparation instructions: [DATA.md](DATA.md).
Default prepared-input folder: `data/revision/deep_starmap_series/` in the repository; set `RAFTUP_TUTORIAL_DATA` to override it. Relative paths inside `pair.json` resolve against that file. Outputs go to `outputs/revision/04_3d_reconstruction/`; set `RAFTUP_OUTPUT_ROOT` to override. The real-data notebooks do not substitute synthetic data when inputs are missing.


In [ ]:
from pathlib import Path
import os, json
import numpy as np
import raftup
from raftup import run_automated_alignment, run_manual_alignment
from raftup.experiments import load_pair, save_alignment, coverage
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), Path.cwd())
DATA = Path(os.environ.get('RAFTUP_TUTORIAL_DATA', str(REPO / 'data/revision/deep_starmap_series'))).expanduser().resolve()
OUT = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', str(REPO / 'outputs/revision/04_3d_reconstruction'))).expanduser().resolve()
print('Inputs:', DATA, '\nOutputs:', OUT)
print('RAFT-UP:', raftup.__version__, raftup.__file__)


In [ ]:
from raftup.experiments import rigid_from_alignment
series_path = DATA / 'series.json'
series = json.loads(series_path.read_text())
Z_SPACING = 10.  # Set to the section spacing in the intended z units.
global_R, global_t = np.eye(2), np.zeros(2)
reconstruction = []
previous_b = None
for pair_index, relative_path in enumerate(series):
    a, b, cost, cfg = load_pair(series_path.parent / relative_path)
    if previous_b is not None:
        assert list(a.obs_names) == list(previous_b.obs_names)
        np.testing.assert_allclose(a.obsm['spatial'], previous_b.obsm['spatial'])
    P, decision = run_automated_alignment(a, b, cost, technology='merfish',
        pixels_per_100um=cfg['pixels_per_100um'])
    R, t = rigid_from_alignment(P, a.obsm['spatial'], b.obsm['spatial'])
    if pair_index == 0:
        reconstruction.append(np.column_stack([a.obsm['spatial'], np.zeros(len(a))]))
    global_t = t @ global_R + global_t
    global_R = R @ global_R
    xy = b.obsm['spatial'] @ global_R + global_t
    reconstruction.append(np.column_stack([xy, np.full(len(b), (pair_index+1)*Z_SPACING)]))
    save_alignment(P, decision, OUT / 'reconstruction' / cfg['name'])
    previous_b = b
OUT.mkdir(parents=True, exist_ok=True)
np.savez(OUT / 'reconstructed_sections.npz', **{f'section_{i:02}':v for i,v in enumerate(reconstruction)})


The saved-result comparison and reference-frame-aware tissue nRMSE used in the paper are in `figures/02_3d_reconstruction.ipynb`. The call above demonstrates package alignment and transform composition; it does not refit to ground truth.
